# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
print("This is an UNSUPERVISED CLUSTERING task.")
print("")
print("Why clustering, not classification?")
print("In Lane 3, I'm discovering natural groupings (archetypes) within the content inventory based on observed behavioral patterns. There is no pre-existing 'true label' that I'm trying to predict.")
print("")
print("What the clustering does:")
print("- Groups similar pages together based on multi-dimensional performance signals (scale, quality, freshness, content type, engagement patterns).")
print("- Reveals recurring archetypes (hidden gems, stale visible pages, rising stars, engagement-problem pages, weak/no-demand pages, champions).")
print("- Each cluster represents a natural behavior pattern that deserves a distinct action playbook.")
print("")
print("Why this beats classification:")
print("With classification, I'd need to define clusters upfront and label training data by hand.")
print("Clustering discovers them from the data itself — no manual labeling required, and it finds patterns humans might not have anticipated.")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
print("In clustering, there is NO target or label — the algorithm discovers structure without supervision.")
print("")
print("Instead, I define a FEATURE SET that captures the dimensions along which pages differ:")
print("")
print("Core observed signals (from the 90-day window):")
print("- Scale: impressions_90d, sessions_90d (log-transformed because traffic is heavy-tailed).")
print("- Quality: ctr, engagement_rate, scroll_rate (what fraction of impressions became clicks/engaged sessions).")
print("- Position: avg_position, position_tier (where pages rank in search results).")
print("- Freshness: days_since_last_update, content_age_days (how fresh is the content).")
print("- Content profile: word_count, content_type, main_intent (article structure and purpose).")
print("- Metadata: competition, search_volume (keyword context).")
print("")
print("What the clustering OUTPUT is:")
print("- Cluster membership: each page gets assigned to one of K clusters.")
print("- Cluster profiles: median/mean values of each feature, revealing each archetype's typical behavior.")
print("- Actionability: each cluster profile drives a specific action (protect, improve, rewrite, prune, monitor).")
print("")
print("Why this is NOT a leakage risk:")
print("- All features come from the same 90-day observation window — no future information sneaks in.")
print("- The output is a grouping strategy, not a causal claim (I'm not claiming 'fixing this will cause recovery').")
print("- I'm ranking pages within each archetype, not predicting an outcome.")

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
print("For clustering, I'll use TWO complementary metrics:")
print("")
print("1. SILHOUETTE SCORE (primary — interpretability):")
print("   - Measures how well each page fits in its assigned cluster vs. others.")
print("   - Range: -1 (bad) to 1 (perfect).")
print("   - Target: >= 0.50 (pages form distinct, tight groups).")
print("   - This tells me: 'Are these archetypes real and separable?'")
print("")
print("2. ACTIONABILITY REVIEW (secondary — business sense):")
print("   - Do the discovered clusters map to real content optimization playbooks?")
print("   - Can I write a distinct action per cluster (e.g., 'refresh this archetype', 'promote that archetype')?")
print("   - Do cluster profiles tell a human story (e.g., 'this cluster is high-impression, low-ctr — title rewrite candidate')?")
print("")
print("Why NOT Precision@K or AUC?")
print("- Those metrics need a binary target (e.g., 'is this page declining'). Clustering has no target.")
print("- Silhouette score directly measures cluster cohesion — the thing I actually care about.")
print("")
print("What 'good' looks like in practice:")
print("- K clusters are discovered (e.g., K=5 or K=7 — I'll test several values).")
print("- Each cluster is interpretable: I can name it (e.g., 'Hidden Gems', 'Stale Visible Pages').")
print("- Cluster profiles differ materially in at least 3 features.")
print("- A content strategist can read the top 3 archetypes and say, 'Yes, I know these types of pages — and yes, they need different fixes.'")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Srujanmp1366/flyrank-internship"
REPO_DIR = "flyrank-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# Load the data
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print(f"Unit of analysis: ONE ROW = ONE PAGE (content item)")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"")
print(f"Shape of the data: {df.shape}")
print(f"")
print("First 5 rows (showing key clustering features):")
print()

# Select relevant columns for clustering
feature_cols = [
    'content_id', 'client_id',
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update',
    'word_count', 'content_type', 'main_intent', 'search_volume', 'competition'
]

available_cols = [c for c in feature_cols if c in df.columns]
print(df[available_cols].head())
print()
print(f"Missing values in key columns:")
print(df[available_cols].isnull().sum())

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
print("Suppose I tried a fixed rule to define archetypes:")
print()
print("Fixed Rule Attempt:")
print("  'If impressions > 1000 AND ctr < 0.5, call it a CTR-problem page.'")
print("  'If impressions < 100 AND ctr > 2, call it a hidden gem.'")
print()
print("Why this breaks:")
print()
print("1. Correlated features hide the real grouping.")
print("   - High-impression pages often have low avg_position (they're ranked high).")
print("   - Low-impression pages often have high avg_position (they're ranked low).")
print("   - So a rule that says 'high impr + low position + high engagement' vs. 'low impr + low engagement' cuts across")
print("     many features at once. A simple if-else tree gets unwieldy.")
print()
print("2. Non-linear relationships.")
print("   - A page with 2,000 impressions and 0.5% CTR is VERY different from one with 100 impressions and 0.5% CTR.")
print("     (The first is a wasted opportunity; the second is just low volume.)")
print("   - Clustering captures this: it learns that SCALE + QUALITY together define archetypes.")
print()
print("3. Interdependencies across multiple dimensions.")
print("   - A page's archetype depends on: impressions, clicks, position, freshness, word count, type, AND engagement.")
print("   - These six dimensions interact. Clustering in 6-D space finds these patterns automatically.")
print("   - A fixed rule would need dozens of if-else branches.")
print()
print("4. Discovering unknown archetypes.")
print("   - A fixed rule can only find archetypes I pre-define.")
print("   - Clustering can find a surprising archetype (e.g., 'pages that rank well but have never been updated').")
print("   - That archetype might not match my intuition — but if it's a coherent cluster, it deserves its own action.")
print()
print("Clustering handles all of this:")
print("- Scales features so each dimension contributes equally (no single metric dominates).")
print("- Learns distance in multi-dimensional space (measures similarity across all signals at once).")
print("- Iterates to find the K best clusters (optimal grouping).")
print("- Returns interpretable profiles (cluster centers) so I can explain each archetype.")
print()
print("Bottom line: ML clustering discovers patterns across many dimensions simultaneously.")
print("Fixed rules can't compete in this space because they require me to pre-enumerate all the patterns.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.